# Advanced Wi-Fi Track Data Clean

This method re-clean data according to track repeat frequency ( caused by unstable signal appears among multiple trackers) based on given basic cleaned data.

Generate **virtual wifi tracker** after cleaning.(Virtual wifi trackers' id will be under zero)

In [33]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mat
import matplotlib.cm as cm
import os
import datetime
import time
from tqdm import tqdm
import plotly.express as px
import plotly.io as pio
import seaborn as sns

import sys
sys.path.append('../plot/')
import myplot

sys.path.append('../utils/')
import utils
import DBSCAN
from DBSCAN import My_DBSCAN,My_DBSCAN_MATRIX

# 设置plotly默认主题
pio.templates.default = 'plotly_white'

mat.rcParams['font.family'] = 'SimHei'
mat.rcParams['font.sans-serif'] = 'SimHei'

import warnings
warnings.filterwarnings('ignore')

from collections import namedtuple

from importlib import reload

## 读取数据->初步清理->保存

In [2]:
df_read = pd.read_csv(os.getcwd()+'/dacang/cleaned_data/dacang_cleaned_data_basic.csv')
df = df_read.copy()
df_read

,a,r,t,m
0,22,-43,2022-07-05 23:49:36,"48,74,38,155,214,98"
1,22,-35,2022-07-05 23:49:36,"252,107,240,89,142,53"
2,22,-42,2022-07-05 23:49:36,"164,125,159,153,152,106"
3,22,-38,2022-07-05 23:49:47,"252,107,240,89,142,53"
4,22,-42,2022-07-05 23:49:47,"164,125,159,153,152,106"
...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"88,102,186,101,140,144"
1916097,66,-49,2022-08-07 23:46:54,"88,102,186,101,140,144"
1916098,66,-48,2022-08-07 23:48:38,"88,102,186,101,140,144"
1916099,66,-49,2022-08-07 23:51:14,"88,102,186,101,140,144"


In [3]:
#label each mac with date
pbar = tqdm(total=len(df))
df['oriMac'] = df['m']
df.t = pd.to_datetime(df.t)
df['calender'] = df.t.apply(lambda x : str(x.month)+'.'+str(x.day))
for index,row in df.iterrows():
    df.loc[index,'m'] = str(row['calender'])+'-'+row.m
    pbar.update(1)
pbar.close()
print("mac length:",len(df.m.unique()))
df

  0%|          | 0/1916101 [00:00<?, ?it/s]

100%|██████████| 1916101/1916101 [02:24<00:00, 13217.61it/s]


,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [5]:
#删除打上标签后只出现过一次的mac
df = df[df.duplicated('m',keep=False)]
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1916096,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916097,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916098,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1916099,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


In [ ]:
#保存
df.to_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv",index=False)

## 读取已打上date标签的dataframe

In [34]:
df = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dataLabeled.csv")
df.t = pd.to_datetime(df.t)
df

,a,r,t,m,oriMac,calender
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5
1,22,-35,2022-07-05 23:49:36,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
2,22,-42,2022-07-05 23:49:36,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
3,22,-38,2022-07-05 23:49:47,"7.5-252,107,240,89,142,53","252,107,240,89,142,53",7.5
4,22,-42,2022-07-05 23:49:47,"7.5-164,125,159,153,152,106","164,125,159,153,152,106",7.5
...,...,...,...,...,...,...
1899913,66,-49,2022-08-07 23:44:30,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899914,66,-49,2022-08-07 23:46:54,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899915,66,-48,2022-08-07 23:48:38,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7
1899916,66,-49,2022-08-07 23:51:14,"8.7-88,102,186,101,140,144","88,102,186,101,140,144",8.7


## 以天为单位清理异常mac

In [35]:
#获取每个mac一天中经过的探针数：a_count
count_list = df.groupby('m').a.value_counts()
df_count = count_list.to_frame().reset_index()
a_count = df_count.m.value_counts()
#获取每个mac一天中的数据量:s_count
count_list = df.m.value_counts()


mac_list = df.m.unique()
len(mac_list)

15615

In [84]:
def GetDuration(df_now):
    t1 = df_now.iloc[0].t
    t2 = df_now.iloc[len(df_now)-1].t
    duration =  t2-t1
    t = (t1 + duration/2)
    moment = t.hour + (t.minute/60)
    duration = duration.components.hours+(duration.components.minutes/60)
    return round(duration,2),round(moment,2)

macDur_dict = {}
macMoment_dict = {}

for mac in tqdm(mac_list):
    df_now = df[df.m == mac]
    duration,moment = GetDuration(df_now)
    macDur_dict.update({mac:duration})
    macMoment_dict.update({mac:moment})

#保存
dur_dict_str = str(macDur_dict)
mom_dict_str = str(macMoment_dict)
with open('dacang/cleaned_date/date_mac_duration_dict.txt', 'w') as file:
   file.write(dur_dict_str)
with open('dacang/cleaned_date/date_mac_moment_dict.txt', 'w') as file:
   file.write(mom_dict_str)

100%|██████████| 15615/15615 [20:13<00:00, 12.86it/s]


In [36]:
#读取
with open('dacang/cleaned_data/date_mac_duration_dict.txt', 'r') as file:
    dict_str = file.read()
    macDur_dict = eval(dict_str)
with open('dacang/cleaned_data/date_mac_moment_dict.txt', 'r') as file:
    dict_str = file.read()
    macMoment_dict = eval(dict_str)
mac_list = list(macDur_dict.keys())
mac_dur_list = list(macDur_dict.values())
mac_moment_list = list(macMoment_dict.values())

In [37]:
#生成统计数值的dataframe
df_sta = pd.DataFrame({'M':[],'Dur':[],'A_count':[],'Count':[],'oriMac':[],'date':[]})
for i in range(len(mac_list)):
    df_sta = df_sta._append({'M':mac_list[i],
                             'Dur':mac_dur_list[i],
                             'Moment':mac_moment_list[i],
                             'A_count':a_count[mac_list[i]],
                             'Count':count_list[mac_list[i]],
                             'oriMac':mac_list[i].split('-')[1],
                             'date':mac_list[i].split('-')[0]},ignore_index=True)
df_sta

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-252,107,240,89,142,53",17.02,2.0,37.0,"252,107,240,89,142,53",7.5,20.33
2,"7.5-164,125,159,153,152,106",17.02,2.0,19.0,"164,125,159,153,152,106",7.5,20.33
3,"7.5-112,138,9,131,175,199",17.02,2.0,47.0,"112,138,9,131,175,199",7.5,20.33
4,"7.5-208,151,254,40,222,169",17.02,3.0,11.0,"208,151,254,40,222,169",7.5,20.33
...,...,...,...,...,...,...,...
15610,"8.7-172,92,102,108,72,204",0.00,1.0,2.0,"172,92,102,108,72,204",8.7,19.02
15611,"8.7-172,186,190,231,156,25",0.00,1.0,2.0,"172,186,190,231,156,25",8.7,19.02
15612,"8.7-216,138,220,202,211,84",1.93,1.0,8.0,"216,138,220,202,211,84",8.7,22.82
15613,"8.7-244,76,112,14,81,63",0.03,1.0,3.0,"244,76,112,14,81,63",8.7,21.88


In [109]:
fig = px.histogram(df_sta.Count.to_list())
fig.show()

- MAC的数据量之间相差及其悬殊，先剔除数据量与预期相差过大的MAC
- 当MAC到达的探针数量<=3时，可视为：
  - 固定设备处于三个探针的信号覆盖范围中，予以剔除
  - 行人移动量过低，视为居家行为，其行为不受环境因素所影响，予以剔除

In [49]:
#--初步清理--#
#剔除一天中数据量小于10且大于10000的mac(小于10000会剔除5个mac)
df_sta = df_sta[df_sta.Count.apply(lambda x : x>10)]
#剔除一天中持续时间小于半小时的mac
df_sta = df_sta[df_sta.Dur>0.5]
#剔除一天中接受探针数小于3的mac
df_sta = df_sta[df_sta.A_count>3]
df_sta = df_sta.reset_index().drop('index',axis = 1)
df_sta

,M,Dur,A_count,Count,oriMac,date,Moment
0,"7.5-48,74,38,155,214,98",21.83,6.0,147.0,"48,74,38,155,214,98",7.5,22.73
1,"7.5-20,178,229,137,47,142",17.02,4.0,38.0,"20,178,229,137,47,142",7.5,20.33
2,"7.6-48,74,38,155,214,98",19.23,12.0,8186.0,"48,74,38,155,214,98",7.6,9.62
3,"7.6-20,178,229,137,47,142",18.90,5.0,2634.0,"20,178,229,137,47,142",7.6,9.45
4,"7.6-4,207,140,11,128,186",23.98,4.0,947.0,"4,207,140,11,128,186",7.6,12.00
...,...,...,...,...,...,...,...
1344,"8.7-176,70,146,156,89,69",9.38,5.0,179.0,"176,70,146,156,89,69",8.7,17.18
1345,"8.7-44,220,173,148,169,213",7.48,4.0,171.0,"44,220,173,148,169,213",8.7,17.82
1346,"8.7-124,161,119,201,50,40",9.78,6.0,260.0,"124,161,119,201,50,40",8.7,18.98
1347,"8.7-152,207,83,118,160,172",19.45,4.0,54.0,"152,207,83,118,160,172",8.7,12.45


## 清除在一段时间内持续在同一地点的mac数据中段

仅保留头尾

部分居民设备信号强度大，导致其数据量明显大于别的mac。
将所有设备一直在同一时间段停留的数据只保留头尾两条

In [ ]:
#!!根据date-Mac过滤主df
df = df[df.m.apply(lambda x : x in df_sta.M.unique().tolist())]
df.to_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_date_dataLabeled_advance_temp.csv",index=False)

In [75]:
#读取过滤后的主df
df = pd.read_csv(os.getcwd()+"/dacang/cleaned_data/dacang_cleaned_data_dateLabeled_advance_temp.csv")
df.t = pd.to_datetime(df.t)
df['mark'] = range(0,len(df))
mac_list = df.m.unique()
df

,a,r,t,m,oriMac,calender,mark
0,22,-43,2022-07-05 23:49:36,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,0
1,22,-43,2022-07-05 23:49:47,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,1
2,22,-43,2022-07-05 23:50:00,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,2
3,22,-35,2022-07-05 23:50:00,"7.5-20,178,229,137,47,142","20,178,229,137,47,142",7.5,3
4,22,-42,2022-07-05 23:50:13,"7.5-48,74,38,155,214,98","48,74,38,155,214,98",7.5,4
...,...,...,...,...,...,...,...
1154356,32,-34,2022-08-07 23:51:18,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7,1154356
1154357,32,-42,2022-08-07 23:51:44,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7,1154357
1154358,32,-27,2022-08-07 23:51:57,"8.7-156,95,90,12,124,73","156,95,90,12,124,73",8.7,1154358
1154359,32,-36,2022-08-07 23:52:49,"8.7-124,161,119,201,50,40","124,161,119,201,50,40",8.7,1154359


In [78]:
def GetRepeatTrack(df_now):
    del_list = []
    track_now = df_now.iloc[0].a
    ind_list = []
    end_mark = df_now.iloc[len(df_now)-1].mark
    for index,row in df_now.iterrows():
        
        if row.a == track_now:
            if row.mark == end_mark:
                if len(ind_list)>2:
                    del_list.extend(ind_list[1:len(ind_list)-1])
            else:
                ind_list.append(row.mark)
        else:
            track_now = row.a
            if len(ind_list)>2:
                del_list.extend(ind_list[1:len(ind_list)-1])
            ind_list = []
    return del_list

del_list = []
for mac in tqdm(mac_list):
    df_now = df[df.m == mac]
    del_list.extend(GetRepeatTrack(df_now))
len(del_list)


100%|██████████| 1349/1349 [01:11<00:00, 18.84it/s]


944295

In [80]:
df_filterd = df[df.mark.apply(lambda x:False if x in del_list else True)]

KeyboardInterrupt: 

# 虚拟探针生成

当行人处在多个设备之间时，信号可能同时被多个设备接收，造成该行人不停移动的假象，也可能使该行人的数据量大幅上升，对后续的离散值去除、聚类都会造成干扰。

将一天内连续在2-3个探针间来回且次数超过一定阈值的轨迹视作停留在这些探针的中间点，并创建

# 清除离群值 - 随机孤立森林

In [111]:
#生成用于处理利群数据的dataframe、并归一化
df_outlier = utils.Normalize_df(df_sta.reindex(columns = ['Dur','A_count','Count','Moment']))
print(len(df_outlier))
df_outlier.head()

1349


,Dur,A_count,Count,Moment
0,9.083546,1.666667,0.137805,9.575552
1,7.033248,0.000000,0.027358,8.556876
2,7.975277,6.666667,8.283514,4.011036
3,7.834612,0.833333,2.657817,3.938879
4,10.000000,0.000000,0.948424,5.021222


In [112]:
reload(myplot)
myplot.Boxes([(df_outlier.Dur,"数据持续时间"),(df_outlier.Count,"数据量"),
              (df_outlier.A_count,"探针数量"),(df_outlier.Moment,"中位时刻")])

- 探针数量受被探测对象的移动范围影响
- 数据量受被探测对象的设备信号强度影响的同时，部分非探测对象设备也可能不间断发出数据，导致数据量激增
因此所经过的探针数量以及数据量不应当被视为噪声评估要素


在排除离群值时仅考虑数据持续时间和中位时刻

In [115]:
from sklearn.ensemble import IsolationForest
rng = np.random.RandomState(42)
model_IF = IsolationForest(max_samples=256,random_state=rng,contamination='auto')
model_IF.fit(df_outlier[['Dur','Count']])
df_outlier['label'] = model_IF.predict(df_outlier[['Dur','Count']])
print(df_outlier.label.value_counts())

fig = px.scatter(df_outlier,y = 'Count',x = 'Dur',color='label')
fig.update_traces(marker_size = 5)
fig.update_layout(scattermode='group')
fig.show()

label
 1    1068
-1     281
Name: count, dtype: int64


In [116]:
df_cluster = df_outlier[df_outlier.label == 1].reset_index().drop('index',axis = 1)
df_cluster = utils.Normalize_df(df_cluster,cols = ['Dur','Moment','Count'])
myplot.Scatter_3D(df_cluster,'Dur','Count','Moment','label',color_name='A_count')

In [124]:
df_result = df_sta[df_outlier.label == 1]
print('清洗后原mac地址的数量:',len(df_result.oriMac.unique()))

清洗后原mac地址的数量: 258


In [ ]:
df_sta[df_outlier.label == 1].to_csv(os.getcwd()+'/dacang/cleaned_data/dacang_dateMac_advance_cleaned.csv',index=False)